In [8]:
# ==========================================
# 1. INSTALL DEPENDENCIES & RESTART (If in Colab)
# ==========================================
%pip install torch_geometric ogb sentence-transformers

You should consider upgrading via the '/home/hice1/rdurai6/scratch/project/graph-semantic-research-main/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [ ]:


import os
import pandas as pd
import numpy as np
import torch
from ogb.nodeproppred import PygNodePropPredDataset
from sentence_transformers import SentenceTransformer

# Fix PyTorch compatibility issues with OGB.PygNodePropPredDataset
if not hasattr(torch, '_is_patched_for_ogb'):
    _orig_load = torch.load
    def patched_load(*args, **kwargs):
        if 'weights_only' not in kwargs:
            kwargs['weights_only'] = False
        return _orig_load(*args, **kwargs)
    torch.load = patched_load
    torch._is_patched_for_ogb = True
    print("Applied PyTorch compatibility fix")


In [10]:
print("Loading OGB graph structure...")
dataset = PygNodePropPredDataset(name='ogbn-arxiv', root='../data/')
data = dataset[0]

# Download raw text
raw_text_url = "https://snap.stanford.edu/ogb/data/misc/ogbn_arxiv/titleabs.tsv.gz"
print(f"Downloading raw text from {raw_text_url}...")
raw_df = pd.read_csv(raw_text_url, sep='\t', compression='gzip', names=['paper id', 'title', 'abstract'], low_memory=False)
print(raw_df.shape)

# Load the local mapping to align graph nodes with text
local_mapping_path = "../data/ogbn_arxiv/mapping/nodeidx2paperid.csv.gz"
mapping_df = pd.read_csv(local_mapping_path, compression='gzip', names=['node index', 'paper id'])
print(mapping_df.shape)

# Merge to ensure the text matches node indices 0 to N
ordered_df = mapping_df.merge(raw_df, on='paper id', how='left')
ordered_df['full_text'] = (ordered_df['title'].fillna('Unknown Title') + ". " + ordered_df['abstract'].fillna('No abstract available.'))
print(ordered_df.shape)

print(f"Sample node text: {ordered_df['full_text'].iloc[5][:150]}...")

Loading OGB graph structure...


(179720, 3)
(169344, 2)
(169344, 5)
Sample node text: analysis of asymptotically optimal sampling based motion planning algorithms for lipschitz continuous dynamical systems. Over the last 20 years signif...


In [11]:
print("Loading SBERT model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

print(f"Encoding {len(ordered_df)} nodes...")

# Using device='cuda' if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
node_features = model.encode(
    ordered_df['full_text'].values, 
    show_progress_bar=True, 
    batch_size=64, 
    device=device
)

print(f"Feature matrix shape: {node_features.shape}")


Loading SBERT model...
Encoding 169344 nodes...


Batches: 100%|██████████| 2646/2646 [03:27<00:00, 12.72it/s]


Feature matrix shape: (169344, 384)


In [12]:
output_path = '../data/arxiv_sbert.npz'
split_idx = dataset.get_idx_split()
num_nodes = data.num_nodes

# Generate Boolean masks from OGB index splits
train_mask = np.zeros(num_nodes, dtype=bool)
train_mask[split_idx['train']] = True
val_mask = np.zeros(num_nodes, dtype=bool)
val_mask[split_idx['valid']] = True
test_mask = np.zeros(num_nodes, dtype=bool)
test_mask[split_idx['test']] = True

# Since ArXiv has 1 fixed split, we repeat it 10 times to match the pipeline.
np.savez(output_path,
         node_features=node_features.astype(np.float32),
         node_labels=data.y.numpy().flatten(),
         edges=data.edge_index.numpy().T,
         train_masks=np.tile(train_mask, (10, 1)),
         val_masks=np.tile(val_mask, (10, 1)),
         test_masks=np.tile(test_mask, (10, 1))
)

print(f"Successfully saved to {output_path}")

Successfully saved to ../data/arxiv_sbert.npz


In [13]:
check = np.load(output_path, allow_pickle=True)
print(f"Keys: {list(check.keys())}")
print(f"Features: {check['node_features'].shape}")
print(f"Labels: {check['node_labels'].shape}")
print(f"Edges: {check['edges'].shape}")
print(f"Masks: {check['train_masks'].shape}")

Keys: ['node_features', 'node_labels', 'edges', 'train_masks', 'val_masks', 'test_masks']
Features: (169344, 384)
Labels: (169343,)
Edges: (1166243, 2)
Masks: (10, 169343)
